# Import Libraries

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [25]:
from datasets import load_dataset
import torch
from torch.utils.data import Dataset, IterableDataset, DataLoader
from torchvision import transforms

https://docs\.pytorch\.org/tutorials/beginner/basics/data\_tutorial\.html

# Data Loaders

In [70]:
# Standardize image size + convert to pytorch tensors
image_size = (224, 224)
default_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor()
])

class RajarshiDataset(Dataset):
    def __init__(self, dataset, transform=default_transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        image = row["Image"]
        label = row["Label_A"]
        if self.transform:
            image = self.transform(image)
        return image, label

class RajarshiDataStream(IterableDataset):
    def __init__(self, dataset, transform=default_transform):
        self.dataset = dataset
        self.transform = transform

    def __iter__(self):
        for row in self.dataset:
            image = self.transform(row["Image"])
            label = row["Label_A"]
            yield image, label

class HemgDataset(Dataset):
    def __init__(self, dataset, transform=default_transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        image = row["image"]
        label = row["label"]
        label = int(1 - label) # need 1 = fake and 0 = real
        if self.transform:
            image = self.transform(image)
        return image, label

class HemgDataStream(IterableDataset):
    def __init__(self, dataset, transform=default_transform):
        self.dataset = dataset
        self.transform = transform

    def __iter__(self):
        for row in self.dataset:
            image = self.transform(row["image"])
            label = row["label"]
            label = int(1 - label) # need 1 = fake and 0 = real
            yield image, label

# See if data loaders work

Testing if the data loader for raj hugging face data set works\.

In [58]:
streaming = True
batch_size = 32
sample_size = 5

train_data_raj = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset", split="train", streaming = streaming).take(sample_size)

if streaming:
    Xtrain_raj = RajarshiDataStream(train_data_raj)
else:
    Xtrain_raj = RajarshiDataset(train_data_raj)

data_loader_raj = DataLoader(Xtrain_raj, batch_size=batch_size)

In [61]:
# test this works
for images, labels in data_loader_raj:
    print("Images", images.shape)
    print("Labels", labels)

Images torch.Size([5, 3, 224, 224])
Labels tensor([0, 1, 1, 1, 1])


In [73]:
train_data_hemg = load_dataset("Hemg/AI-Generated-vs-Real-Images-Datasets", split="train", streaming = streaming).take(sample_size)

if streaming:
    Xtrain_hemg = HemgDataStream(train_data_hemg)
else:
    Xtrain_hemg = HemgDataset(train_data_hemg)

data_loader_hemg = DataLoader(Xtrain_hemg, batch_size=batch_size)

Testing if the data loader for hemg hugging face worked

In [76]:
for images, labels in data_loader_hemg:
    print("Images", images.shape)
    print("Labels", labels)

Images torch.Size([5, 3, 224, 224])
Labels tensor([1, 1, 1, 1, 1])


How to interpret:

<img src="Screenshot 2026-03-15 at 5.17.46â¯PM.png" width="" align="" />

Images Torch Size \[ \# images, 3 channels RGB, 224 pixels width x 224 pixels width\]
Labels = \[1, 1, 1, 1, 1,\] means all 5 images in this batch of 5 are fake AI\-generated\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ca50555b-201c-448d-974c-16107e326479' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>